# ParSNIP-first transient classifier

Required packages are imported directly from the active kernel. The editable
WarpTemplate installation anchors all samples and generated artifacts in the shared
`data` directory beside the repository. The `USER SETTINGS` cell selects one readable `RUN_NAME` and an explicit
action for each expensive stage. Missing inputs fail at their fixed path; no alternate
location is searched.


In [ ]:
# Configure paths and import the installed ParSNIP package plus the local workflow.
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import parsnip
import seaborn as sns
from astropy.table import Table

import warptemplate
from warptemplate import classification as workflow

# Anchor shared data beside the editable checkout, independent of the kernel cwd.
DATA_ROOT = Path(warptemplate.__file__).resolve().parents[2] / "data"
sns.set_theme(context="notebook", style="ticks")


## Configuration

Use `run` to create a new stage, `load` to read its fixed artifacts directly, and
`skip` to leave it untouched. ParSNIP deliberately has no resume action because its
native checkpoint does not preserve optimizer, scheduler, epoch, or RNG state. Use a
new `RUN_NAME` instead of overwriting a completed scientific experiment.


In [ ]:
# Define user settings first, then derive every fixed artifact path.
# ==============================================================================
# USER SETTINGS — EDIT VALUES HERE, THEN SELECT "RUN ALL"
# ==============================================================================
SAMPLE_ID = "warp_sample_combined_schema6_660649e34711"
RUN_NAME = "parsnip_baseline"
SPLIT_STRATEGY = "basis_sn"
SEED = 20260721
DEVICE = "cpu"
THREADS = 14
FULL_EPOCHS = 100
MIN_CHILD_WEIGHTS = [1, 10, 30]

# Each stage is explicit. ParSNIP intentionally has no resume action because its
# checkpoint omits optimizer, scheduler, epoch, and random-number-generator state.
SPLIT_ACTION = "load"                 # run, load
SMOKE_ACTION = "run"                 # run, load, skip
PARSNIP_MODEL_ACTION = "skip"         # run, load, skip
REPRESENTATION_ACTION = "skip"        # run, load, skip
CLASSIFIER_ACTION = "skip"            # run, load, skip
TEST_ACTION = "skip"                  # run, load, skip

# ==============================================================================
# DERIVED CONFIGURATION — DO NOT EDIT PATHS BELOW
# ==============================================================================
SAMPLE_DIR = DATA_ROOT / "training_samples" / SAMPLE_ID
SPLIT_ROOT = DATA_ROOT / "classification_splits" / SAMPLE_ID
RUN_DIR = (
    DATA_ROOT / "classifier_runs" / SAMPLE_ID / SPLIT_STRATEGY
    / "parsnip" / RUN_NAME
)
EXPERIMENT_PATH = RUN_DIR / "experiment.json"
MODEL_PATH = RUN_DIR / "models" / "parsnip.pt"
CLASSIFIER_PATH = RUN_DIR / "models" / "lightgbm.pkl"
FIGURE_DIR = RUN_DIR / "figures"
FULL_HISTORY_PATH = RUN_DIR / "histories" / "parsnip_training.json"

SMOKE_CONFIG = {
    "epochs": 2,
    "objects_per_class": 64,
    "train_folds": list(range(4, 10)),
    "validation_fold": 3,
    "test_fold": 2,
    "seed": SEED,
    "bands": list(workflow.EXPECTED_BANDS),
    "lightgbm_min_child_weight_grid": MIN_CHILD_WEIGHTS,
}
from parsnip.settings import default_settings as parsnip_default_settings
MODEL_CONFIG = {
    "max_epochs": FULL_EPOCHS,
    "threads": THREADS,
    "bands": list(workflow.EXPECTED_BANDS),
    "parsnip_settings": {
        key: value for key, value in parsnip_default_settings.items() if value is not None
    },
    "lightgbm_min_child_weight_grid": MIN_CHILD_WEIGHTS,
    "smoke": SMOKE_CONFIG,
}
CONFIG = workflow.ExperimentConfig(
    training_sample=SAMPLE_ID,
    evaluation_sample=SAMPLE_ID,
    backend="parsnip",
    redshift_mode="truth_z",
    split_strategy=SPLIT_STRATEGY,
    seed=SEED,
    model_config=MODEL_CONFIG,
    run_id=RUN_NAME,
)
workflow.set_random_seed(SEED)
print(json.dumps(CONFIG.normalized(), indent=2))
print(f"Input:  {SAMPLE_DIR}")
print(f"Output: {RUN_DIR}")


## Load, audit, and persist the split

Objects with fewer than three 0.33-day grouped observing epochs are excluded before splitting. The manifest is object-level and records both raw and merged labels, provenance IDs, fold, partition, strategy, and seed. Photometry remains only in the source observation tree.

In [ ]:
# Audit the schema-6 ensemble without loading 17 million observations at once.
# Truth has one row per simulated object; the observation audit streams the much larger photometry tree.
truth = workflow.load_sample_truth(SAMPLE_DIR)
source_manifest = workflow.load_sample_manifest(SAMPLE_DIR)
observation_audit = workflow.audit_sample_observations(SAMPLE_DIR)
# Merge only SLSN-I and SLSN-II; fitclass remains available as the unmodified raw label.
truth["final_label"] = workflow.merge_fitclasses(truth["fitclass"]).to_numpy()
print(f"Truth objects: {len(truth):,}; observation rows: {observation_audit['observation_rows']:,}")
print(f"Cadence realizations: {truth['survey_realization_id'].nunique()}")
display(pd.crosstab(truth["fitclass"], truth["final_label"], margins=True))
display(pd.Series(observation_audit["band_counts"], name="rows").reindex(workflow.EXPECTED_BANDS).to_frame())
# These assertions turn schema or band mismatches into an immediate failure before training.
assert set(truth["final_label"]) == set(workflow.FINAL_CLASSES)
assert set(observation_audit["band_counts"]) == set(workflow.EXPECTED_BANDS)
assert observation_audit["invalid_rows"] == 0

In [ ]:
# Create or directly load the one selected leakage-safe split.
split_manifest, excluded = workflow.prepare_persistent_split(
    SAMPLE_DIR,
    SPLIT_ROOT,
    strategy=SPLIT_STRATEGY,
    seed=SEED,
    action=SPLIT_ACTION,
)
workflow.validate_split_manifest(split_manifest)
print(f"Excluded objects: {len(excluded)}")
display(excluded)
display(pd.crosstab(split_manifest["final_label"], split_manifest["split"], margins=True))
display(split_manifest.groupby("split")["group_id"].nunique().rename("groups"))


## Training-only diagnostics

Cadence, signal-to-noise ratio (S/N), and example light curves are inspected only in the training partition. This is the same discipline as not looking at an exam before choosing how to study: test-set patterns must not influence model configuration.

In [ ]:
# Plot bounded training-only diagnostics without materializing the full ensemble.
# Diagnostics use training objects only, because even exploratory knowledge of test behavior can bias choices.
diagnostic_ids = workflow.sample_balanced_object_ids(
    split_manifest, per_class=512, seed=SEED + 10
)
# Load photometry only for the selected IDs; this keeps memory independent of the full 17-million-row tree.
training_observations = workflow.load_sample_observations(
    SAMPLE_DIR, object_ids=diagnostic_ids
)
# Reduce each light curve to cadence, epoch-count, SNR, and coverage summaries, then attach its class.
training_summary = workflow.summarize_light_curves(training_observations).merge(
    split_manifest[["object_id", "final_label"]], on="object_id"
)
# ECDFs show the complete distribution rather than hiding sparse tails behind a single mean.
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.ecdfplot(data=training_summary, x="median_cadence_days", hue="final_label", ax=axes[0])
sns.ecdfplot(data=training_summary, x="median_snr", hue="final_label", ax=axes[1], legend=False)
axes[0].set(xlabel="Median cadence [days]", xscale="log")
axes[1].set(xlabel="Median |flux| / flux error", xscale="log")
figure.tight_layout()
display(training_summary.groupby("final_label")[["observations", "epochs", "median_snr"]].median())

In [ ]:
# Display one representative training light curve per final class.
# Zeropoint conversion changes numerical flux units but preserves AB magnitude and signal-to-noise.
plot_observations = workflow.rescale_flux_to_zeropoint(training_observations)
# Choose the object nearest the class-median observation count, avoiding a subjective hand-picked example.
representative_ids = {}
for label, group in training_summary.groupby("final_label"):
    target_count = group["observations"].median()
    representative_ids[label] = group.iloc[(group["observations"] - target_count).abs().argmin()]["object_id"]
figure, axes = plt.subplots(4, 2, figsize=(14, 14), sharex=False)
# Plot every survey band separately so cadence gaps and ZTF/LSST coverage remain visible.
for axis, (label, object_id) in zip(axes.flat, representative_ids.items()):
    light_curve = plot_observations[plot_observations["object_id"] == object_id]
    for band, band_rows in light_curve.groupby("band"):
        axis.errorbar(band_rows["mjd"], band_rows["flux"], yerr=band_rows["fluxerr"], fmt=".", label=band)
    axis.set(title=label, xlabel="MJD", ylabel="Flux (zeropoint 25)")
    axis.legend(ncol=3, fontsize=7)
for axis in axes.flat[len(representative_ids):]:
    axis.set_visible(False)
figure.tight_layout()

## Convert to `lcdata`

ParSNIP assumes fluxes use zeropoint 25. Each flux and uncertainty is multiplied by `10**((25 - zp) / 2.5)`, preserving AB magnitude and S/N. `mjd` becomes `time`, merged `fitclass` becomes `type`, and exact simulated `z` becomes `redshift`. All nine ZTF/LSST bands remain distinct.

In [ ]:
# Materialize full lcdata partitions only for a newly requested ParSNIP fit.
train_dataset = None
validation_dataset = None
if PARSNIP_MODEL_ACTION == "run":
    train_dataset = workflow.dataset_for_partition(
        truth, SAMPLE_DIR, split_manifest, "train"
    )
    validation_dataset = workflow.dataset_for_partition(
        truth, SAMPLE_DIR, split_manifest, "validation"
    )
    print(
        f"lcdata sizes: train={len(train_dataset):,}, "
        f"validation={len(validation_dataset):,}"
    )
else:
    print("Full lcdata materialization is not needed for this action.")


## Two-epoch smoke test and CPU timing

The smoke experiment is a complete miniature classifier workflow. It draws 64 objects per class from folds 4–9 for training, fold 3 for validation, and fold 2 for smoke testing. These remain provenance-group separated. ParSNIP trains for two epochs, LightGBM selects `min_child_weight` on smoke validation, and the refitted classifier produces validation and smoke-test confusion matrices. The official fold-0 test set remains untouched, so this result is for workflow verification rather than final scientific reporting. Because ParSNIP internally repeats augmented views for small datasets, it is still a meaningful CPU job. The measured wall time is scaled into a rough full-run estimate.

In [ ]:
# Run, load, or skip the disposable end-to-end ParSNIP smoke experiment.
from IPython.display import Image
from sklearn.metrics import confusion_matrix

smoke_output = RUN_DIR / "smoke"
smoke_path = smoke_output / "parsnip.pt"
smoke_classifier_path = smoke_output / "lightgbm.pkl"
smoke_history_path = smoke_output / "history.json"
smoke_validation_grid_path = smoke_output / "validation_grid.parquet"
smoke_validation_prediction_path = smoke_output / "validation_predictions.parquet"
smoke_test_prediction_path = smoke_output / "test_predictions.parquet"
smoke_validation_metric_path = smoke_output / "validation_metrics.json"
smoke_test_metric_path = smoke_output / "test_metrics.json"
smoke_figure_path = smoke_output / "confusion_matrices.png"

smoke_history = None
if SMOKE_ACTION == "run":
    workflow.ensure_run_metadata(CONFIG, split_manifest, EXPERIMENT_PATH)
    smoke_output.mkdir(parents=True, exist_ok=False)
    smoke_train_ids = workflow.sample_balanced_object_ids(
        split_manifest,
        per_class=SMOKE_CONFIG["objects_per_class"],
        seed=SEED,
        folds=SMOKE_CONFIG["train_folds"],
    )
    smoke_validation_ids = workflow.sample_balanced_object_ids(
        split_manifest,
        per_class=SMOKE_CONFIG["objects_per_class"],
        seed=SEED + 1,
        folds=[SMOKE_CONFIG["validation_fold"]],
    )
    smoke_test_ids = workflow.sample_balanced_object_ids(
        split_manifest,
        per_class=SMOKE_CONFIG["objects_per_class"],
        seed=SEED + 2,
        folds=[SMOKE_CONFIG["test_fold"]],
    )
    smoke_group_sets = {
        name: set(split_manifest.loc[split_manifest["object_id"].isin(ids), "group_id"])
        for name, ids in {
            "train": smoke_train_ids,
            "validation": smoke_validation_ids,
            "test": smoke_test_ids,
        }.items()
    }
    assert not smoke_group_sets["train"] & smoke_group_sets["validation"]
    assert not smoke_group_sets["train"] & smoke_group_sets["test"]
    assert not smoke_group_sets["validation"] & smoke_group_sets["test"]

    smoke_datasets = {
        "smoke_train": workflow.to_lcdata(truth, SAMPLE_DIR, smoke_train_ids),
        "smoke_validation": workflow.to_lcdata(
            truth, SAMPLE_DIR, smoke_validation_ids
        ),
        "smoke_test": workflow.to_lcdata(truth, SAMPLE_DIR, smoke_test_ids),
    }
    smoke_model = parsnip.ParsnipModel(
        str(smoke_path),
        list(workflow.EXPECTED_BANDS),
        device=DEVICE,
        threads=THREADS,
    )
    started = time.perf_counter()
    smoke_model.fit(
        smoke_datasets["smoke_train"],
        max_epochs=SMOKE_CONFIG["epochs"],
        augment=True,
        test_dataset=smoke_datasets["smoke_validation"],
    )
    smoke_seconds = time.perf_counter() - started
    # Released astro-parsnip exposes its epoch counter, but no per-epoch history.
    smoke_history = {
        "elapsed_seconds": smoke_seconds,
        "epoch_counter": int(smoke_model.epoch),
        "requested_max_epochs": int(SMOKE_CONFIG["epochs"]),
    }
    workflow.write_json_once(smoke_history, smoke_history_path)

    reloaded_smoke = parsnip.load_model(
        str(smoke_path), device=DEVICE, threads=THREADS
    )
    smoke_representations = {
        name: reloaded_smoke.predict_dataset(dataset)
        for name, dataset in smoke_datasets.items()
    }
    for name, representation in smoke_representations.items():
        workflow.write_table_once(
            representation.to_pandas(), smoke_output / f"representations_{name}.parquet"
        )

    selected_smoke_classifier, smoke_validation_grid = workflow.tune_parsnip_classifier(
        smoke_representations["smoke_train"],
        smoke_representations["smoke_validation"],
        MIN_CHILD_WEIGHTS,
    )
    best_smoke_weight = float(smoke_validation_grid.iloc[0]["min_child_weight"])
    raw_smoke_validation = selected_smoke_classifier.classify(
        smoke_representations["smoke_validation"]
    )
    final_smoke_classifier = workflow.refit_parsnip_classifier(
        smoke_representations["smoke_train"],
        smoke_representations["smoke_validation"],
        best_smoke_weight,
    )
    workflow.write_parsnip_classifier_once(
        final_smoke_classifier, smoke_classifier_path
    )
    raw_smoke_test = final_smoke_classifier.classify(
        smoke_representations["smoke_test"]
    )

    smoke_manifest = split_manifest[
        split_manifest["object_id"].isin(
            smoke_train_ids + smoke_validation_ids + smoke_test_ids
        )
    ].copy()
    smoke_manifest["split"] = "smoke_train"
    smoke_manifest.loc[
        smoke_manifest["object_id"].isin(smoke_validation_ids), "split"
    ] = "smoke_validation"
    smoke_manifest.loc[
        smoke_manifest["object_id"].isin(smoke_test_ids), "split"
    ] = "smoke_test"
    smoke_validation_predictions = workflow.standardize_predictions(
        raw_smoke_validation, smoke_manifest, CONFIG, partition="smoke_validation"
    )
    smoke_test_predictions = workflow.standardize_predictions(
        raw_smoke_test, smoke_manifest, CONFIG, partition="smoke_test"
    )
    smoke_validation_metrics = workflow.compute_classification_metrics(
        smoke_validation_predictions
    )
    smoke_test_metrics = workflow.compute_classification_metrics(smoke_test_predictions)
    workflow.write_table_once(smoke_validation_grid, smoke_validation_grid_path)
    workflow.write_table_once(
        smoke_validation_predictions, smoke_validation_prediction_path
    )
    workflow.write_table_once(smoke_test_predictions, smoke_test_prediction_path)
    workflow.write_json_once(smoke_validation_metrics, smoke_validation_metric_path)
    workflow.write_json_once(smoke_test_metrics, smoke_test_metric_path)

    figure, axes = plt.subplots(1, 2, figsize=(15, 6))
    for axis, title, predictions in (
        (axes[0], "Smoke validation", smoke_validation_predictions),
        (axes[1], "Smoke test", smoke_test_predictions),
    ):
        matrix = confusion_matrix(
            predictions["true_class"],
            predictions["predicted_class"],
            labels=workflow.FINAL_CLASSES,
            normalize="true",
        )
        sns.heatmap(
            matrix,
            annot=True,
            fmt=".2f",
            vmin=0,
            vmax=1,
            cmap="Blues",
            xticklabels=workflow.FINAL_CLASSES,
            yticklabels=workflow.FINAL_CLASSES,
            ax=axis,
        )
        axis.set(title=title, xlabel="Predicted class", ylabel="True class")
    figure.tight_layout()
    figure.savefig(smoke_figure_path, dpi=180, bbox_inches="tight")

elif SMOKE_ACTION == "load":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    smoke_history = json.loads(smoke_history_path.read_text())
    smoke_validation_grid = pd.read_parquet(smoke_validation_grid_path)
    smoke_validation_predictions = pd.read_parquet(smoke_validation_prediction_path)
    smoke_test_predictions = pd.read_parquet(smoke_test_prediction_path)
    smoke_validation_metrics = json.loads(smoke_validation_metric_path.read_text())
    smoke_test_metrics = json.loads(smoke_test_metric_path.read_text())
elif SMOKE_ACTION != "skip":
    raise ValueError("SMOKE_ACTION must be 'run', 'load', or 'skip'")

if SMOKE_ACTION != "skip":
    display(Image(filename=str(smoke_figure_path)))
    display(smoke_validation_grid)
    display(
        pd.DataFrame(
            {
                "validation": {
                    key: smoke_validation_metrics[key]
                    for key in (
                        "class_balanced_log_loss", "balanced_accuracy", "macro_f1",
                        "top_1_accuracy", "top_2_accuracy",
                    )
                },
                "smoke_test": {
                    key: smoke_test_metrics[key]
                    for key in (
                        "class_balanced_log_loss", "balanced_accuracy", "macro_f1",
                        "top_1_accuracy", "top_2_accuracy",
                    )
                },
            }
        )
    )
    print(f"Smoke training time: {smoke_history['elapsed_seconds'] / 60:.1f} min")
else:
    print("Smoke stage skipped.")


## Full ParSNIP training and representations

The full generative model sees training objects only and uses ParSNIP's built-in augmentation. Validation loss is diagnostic and may affect training duration, but test never participates. Representations are then generated for each partition. Do not inspect test representations until the classifier and experiment metadata have been frozen.

In [ ]:
# Run, directly load, or skip the full ParSNIP representation model.
model = None
if PARSNIP_MODEL_ACTION == "run":
    workflow.ensure_run_metadata(CONFIG, split_manifest, EXPERIMENT_PATH)
    model = parsnip.ParsnipModel(
        str(MODEL_PATH),
        list(workflow.EXPECTED_BANDS),
        device=DEVICE,
        threads=THREADS,
    )
    started = time.perf_counter()
    model.fit(
        train_dataset,
        max_epochs=FULL_EPOCHS,
        augment=True,
        test_dataset=validation_dataset,
    )
    # Released astro-parsnip exposes its epoch counter, but no per-epoch history.
    training_history = {
        "elapsed_seconds": time.perf_counter() - started,
        "epoch_counter": int(model.epoch),
        "requested_max_epochs": int(FULL_EPOCHS),
    }
    workflow.write_json_once(training_history, FULL_HISTORY_PATH)
elif PARSNIP_MODEL_ACTION == "load":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    model = parsnip.load_model(str(MODEL_PATH), device=DEVICE, threads=THREADS)
    training_history = json.loads(FULL_HISTORY_PATH.read_text())
    print(f"Training time: {training_history['elapsed_seconds'] / 3600:.2f} h")
elif PARSNIP_MODEL_ACTION != "skip":
    raise ValueError("PARSNIP_MODEL_ACTION must be 'run', 'load', or 'skip'")
else:
    print("Full ParSNIP model stage skipped.")


In [ ]:
# Run, directly load, or skip deterministic ParSNIP representations.
representation_paths = {
    split: RUN_DIR / "representations" / f"{split}.parquet"
    for split in ("train", "validation", "test")
}
representations = {}
if REPRESENTATION_ACTION == "run":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    representation_model = parsnip.load_model(
        str(MODEL_PATH), device=DEVICE, threads=THREADS
    )
    for split, output_path in representation_paths.items():
        dataset = workflow.dataset_for_partition(
            truth, SAMPLE_DIR, split_manifest, split
        )
        table = representation_model.predict_dataset(dataset, augment=False)
        representations[split] = table
        workflow.write_table_once(table.to_pandas(), output_path)
elif REPRESENTATION_ACTION == "load":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    representations = {
        split: Table.from_pandas(pd.read_parquet(output_path))
        for split, output_path in representation_paths.items()
    }
elif REPRESENTATION_ACTION != "skip":
    raise ValueError("REPRESENTATION_ACTION must be 'run', 'load', or 'skip'")
else:
    print("Representation stage skipped.")


## Validation selection, refit, and freeze

LightGBM is the supervised classifier on ParSNIP's representation. Inverse-frequency weights make each class contribute equal total training weight. We compare `min_child_weight = 1, 10, 30` using class-balanced validation log loss, then refit that one choice on training plus validation. The untouched test set is still not scored.

In [ ]:
# Run, directly load, or skip LightGBM selection and the frozen classifier.
VALIDATION_GRID_PATH = RUN_DIR / "metrics" / "validation_grid.parquet"
CLASSIFIER_METADATA_PATH = RUN_DIR / "models" / "lightgbm.json"
classifier = None
validation_scores = None
if CLASSIFIER_ACTION == "run":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    train_representation = Table.from_pandas(
        pd.read_parquet(representation_paths["train"])
    )
    validation_representation = Table.from_pandas(
        pd.read_parquet(representation_paths["validation"])
    )
    _, validation_scores = workflow.tune_parsnip_classifier(
        train_representation, validation_representation, MIN_CHILD_WEIGHTS
    )
    best_weight = float(validation_scores.iloc[0]["min_child_weight"])
    classifier = workflow.refit_parsnip_classifier(
        train_representation, validation_representation, best_weight
    )
    workflow.write_parsnip_classifier_once(classifier, CLASSIFIER_PATH)
    workflow.write_table_once(validation_scores, VALIDATION_GRID_PATH)
    workflow.write_json_once(
        {"selected_min_child_weight": best_weight}, CLASSIFIER_METADATA_PATH
    )
elif CLASSIFIER_ACTION == "load":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    classifier = parsnip.Classifier.load(str(CLASSIFIER_PATH))
    validation_scores = pd.read_parquet(VALIDATION_GRID_PATH)
    classifier_metadata = json.loads(CLASSIFIER_METADATA_PATH.read_text())
elif CLASSIFIER_ACTION != "skip":
    raise ValueError("CLASSIFIER_ACTION must be 'run', 'load', or 'skip'")
else:
    print("Classifier stage skipped.")
if validation_scores is not None:
    display(validation_scores)


## One-time test evaluation

Primary metrics are class-balanced log loss, balanced accuracy, and macro-F1. Per-class precision/recall/F1 exposes which classes fail; top-2 accuracy shows whether the correct class remains among plausible alternatives. The multiclass Brier score evaluates the full probability vector, while calibration compares stated confidence with empirical correctness. Confidence intervals resample the active provenance groups, not individual correlated light curves.

In [ ]:
# Run once, directly load, or skip the immutable official test evaluation.
PREDICTION_PATH = RUN_DIR / "predictions" / "test.parquet"
TEST_METRIC_PATH = RUN_DIR / "metrics" / "test.json"
TEST_BREAKDOWN_PATH = RUN_DIR / "metrics" / "test_breakdowns.parquet"
prediction_table = None
metrics = None
if TEST_ACTION == "run":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    frozen_classifier = parsnip.Classifier.load(str(CLASSIFIER_PATH))
    test_representation = Table.from_pandas(
        pd.read_parquet(representation_paths["test"])
    )
    representations["test"] = test_representation
    raw_classifications = frozen_classifier.classify(test_representation)
    prediction_table = workflow.standardize_predictions(
        raw_classifications, split_manifest, CONFIG, partition="test"
    )
    metrics = workflow.compute_classification_metrics(prediction_table)
    metrics["group_bootstrap_95"] = workflow.group_bootstrap_confidence_intervals(
        prediction_table, split_manifest, repeats=1000, seed=SEED
    )
    test_observations = workflow.select_partition_rows(
        SAMPLE_DIR, split_manifest, "test"
    )
    breakdowns = workflow.metric_breakdowns(
        prediction_table, truth, test_observations
    )
    workflow.write_table_once(prediction_table, PREDICTION_PATH)
    workflow.write_json_once(metrics, TEST_METRIC_PATH)
    workflow.write_table_once(breakdowns, TEST_BREAKDOWN_PATH)
elif TEST_ACTION == "load":
    workflow.load_run_metadata(EXPERIMENT_PATH, CONFIG)
    prediction_table = pd.read_parquet(PREDICTION_PATH)
    metrics = json.loads(TEST_METRIC_PATH.read_text())
    breakdowns = pd.read_parquet(TEST_BREAKDOWN_PATH)
    representations["test"] = Table.from_pandas(
        pd.read_parquet(representation_paths["test"])
    )
elif TEST_ACTION != "skip":
    raise ValueError("TEST_ACTION must be 'run', 'load', or 'skip'")
else:
    print("Official test stage skipped.")
if metrics is not None:
    display(
        pd.Series(
            {key: value for key, value in metrics.items() if isinstance(value, (int, float))}
        )
    )


## Post-evaluation analysis

These plots are intentionally gated on an existing frozen test prediction file. Predictive entropy is high when probability is spread across classes. Error tables should be read together with redshift, cadence, S/N, and survey coverage: they help identify failure regimes, but they must not be used to retune this already evaluated run. Any subsequent change is a new run ID.

In [ ]:
# Plot latent space, confusion matrices, calibration, uncertainty, and difficult cases.
# Nothing is plotted until official predictions exist, preventing accidental visual tuning on fold 0.
if prediction_table is not None:
    from sklearn.metrics import confusion_matrix

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    # Join latent coordinates to truth and predictions using object_id, never row position.
    test_representation = representations["test"].to_pandas()
    latent = test_representation.merge(
        prediction_table[["object_id", "true_class", "predicted_class"]], on="object_id"
    )
    probability_columns = [f"prob_{label}" for label in workflow.FINAL_CLASSES]
    probabilities = prediction_table[probability_columns].to_numpy()
    # Entropy is high for diffuse probabilities; clipping avoids log(0) without changing meaningful values.
    prediction_table["predictive_entropy"] = -(probabilities * np.log(np.clip(probabilities, 1e-15, 1))).sum(axis=1)

    figure, axes = plt.subplots(2, 2, figsize=(14, 11))
    sns.scatterplot(data=latent, x="s1", y="s2", hue="true_class", s=25, alpha=0.7, ax=axes[0, 0])
    # Raw counts expose sample size; row normalization makes the diagonal equal per-class recall.
    raw_confusion = confusion_matrix(prediction_table["true_class"], prediction_table["predicted_class"], labels=workflow.FINAL_CLASSES)
    normalized_confusion = confusion_matrix(prediction_table["true_class"], prediction_table["predicted_class"], labels=workflow.FINAL_CLASSES, normalize="true")
    sns.heatmap(raw_confusion, annot=True, fmt="d", xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES, ax=axes[0, 1])
    sns.heatmap(normalized_confusion, annot=True, fmt=".2f", vmin=0, vmax=1, xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES, ax=axes[1, 0])
    # A calibrated model should lie near y=x: stated confidence then matches empirical accuracy.
    calibration = pd.DataFrame(metrics["calibration"])
    axes[1, 1].plot([0, 1], [0, 1], "k--", label="Ideal")
    axes[1, 1].plot(calibration["confidence"], calibration["accuracy"], "o-", label="Model")
    axes[1, 1].set(xlabel="Mean confidence", ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
    axes[1, 1].legend()
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "latent_confusion_calibration.png", dpi=180)

    # Rank wrong predictions by uncertainty and attach observing conditions for scientific error inspection.
    test_features = workflow.summarize_light_curves(workflow.select_partition_rows(SAMPLE_DIR, split_manifest, "test"))
    errors = prediction_table.merge(test_features, on="object_id")
    errors = errors[errors["true_class"] != errors["predicted_class"]].sort_values("predictive_entropy", ascending=False)
    display(errors[["object_id", "true_class", "predicted_class", "predictive_entropy", "median_snr", "median_cadence_days", "coverage"]].head(25))
else:
    print("Post-evaluation plots remain disabled until frozen test predictions exist.")

In [ ]:
# Inspect reconstructions for representative evaluated objects after test is frozen.
# These plots diagnose the generative ParSNIP model separately from LightGBM classification errors.
if prediction_table is not None and model is not None:
    test_dataset = workflow.dataset_for_partition(truth, SAMPLE_DIR, split_manifest, "test")
    # Key by object_id so reconstructed curves cannot be mismatched through changing table order.
    object_lookup = {curve.meta["object_id"]: curve for curve in test_dataset.light_curves}
    # Select deterministically rather than choosing visually attractive examples after seeing the curves.
    reconstruction_ids = prediction_table.sort_values("object_id").groupby("true_class").first()["object_id"].tolist()
    figure, axes = plt.subplots(4, 2, figsize=(14, 14))
    for axis, object_id in zip(axes.flat, reconstruction_ids):
        observed = object_lookup[object_id]
        # sample=False plots the posterior mean reconstruction rather than a random latent realization.
        model_times, model_flux, _ = model.predict_light_curve(observed, sample=False)
        for band in np.unique(observed["band"]):
            observed_band = observed[observed["band"] == band]
            axis.errorbar(observed_band["time"], observed_band["flux"], yerr=observed_band["fluxerr"], fmt=".", alpha=0.7)
            band_index = model.settings["bands"].index(band)
            axis.plot(model_times, model_flux[0, band_index], alpha=0.8, label=band)
        axis.set_title(object_id.rsplit(":", 2)[-2])
    for axis in axes.flat[len(reconstruction_ids):]:
        axis.set_visible(False)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "representative_reconstructions.png", dpi=180)

## Next comparisons

Treat this exact-redshift ParSNIP run as one cell in a larger experiment matrix. Future samples and backends must reuse the same frozen evaluation groups. Runs are directly comparable only when both `evaluation_sample` and `split_strategy` match. A photometry-only ParSNIP design and the two SuperNNova redshift variants should receive new run IDs rather than changing this baseline. The complete SuperNNova implementation prompt is saved beside this notebook in `SUPERNNOVA_IMPLEMENTATION_PROMPT.md`.